# MTCA-2 · Stage 9b · Unframed-Baseline Test (H10)

**Purpose:** test the H10 prediction. Stage 8.5 had the models *predict* their unframed behavior ("a synthesis of both"). Stage 9 collected the actual open unframed responses. Stage 9b measures where each unframed read sits relative to that model's own F0 / F1_clinical / F2_metaphysical for the same principle.

**Method:** embed each unframed response (`all-MiniLM-L6-v2`); compute cosine to own F0/F1/F2; classify balanced (synthesis) vs lean by `|cos_F1 − cos_F2|` against a threshold. Continuous `mean(F1−F2)` and a threshold-sensitivity table are reported so the finding does not hinge on the cutoff.

**No new API calls.** Deterministic re-analysis of committed Stage 6 + Stage 9 responses (64/65 unframed clean; SSP_P10/deepseek excluded — one structural parse failure at collection).

**Result (see S9b-3):** partial disconfirmation. Balanced fraction is threshold-dependent (33%→64% across ±0.02–0.05), so the robust findings are the near-zero population mean and the per-model signatures, not the exact synthesis percentage.


## S9b-0 — Load committed Stage 6 + Stage 9 responses

In [ ]:
# S9b-0
!pip -q install sentence-transformers

import os, re, json, hashlib, sys, subprocess
from pathlib import Path
from datetime import datetime, timezone
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
REPO = Path("/content/mtca-research") if IN_COLAB else Path("./mtca-research")
if not REPO.exists():
    subprocess.run(["git","clone","https://github.com/billyrdavis1985-bot/mtca-research.git",str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)

MTCA2=REPO/"mtca-2"; S6=MTCA2/"stage6_execution"/"responses"; S9=MTCA2/"stage9_unframed"/"responses"

def extract_framed(r):
    p=r.get('parsed_json',{})
    if not p: return ''
    parts=[]
    for f in ['communicative_function','coherence_assessment']:
        v=p.get(f,'')
        if isinstance(v,str): parts.append(v)
    for f in ['implicit_claims','conditions_for_usefulness','conditions_for_misleading']:
        v=p.get(f,[])
        if isinstance(v,list): parts.extend(str(x) for x in v)
        elif isinstance(v,str): parts.append(v)
    return ' '.join(parts)

def extract_unframed(r):
    p=r.get('parsed_json',{})
    if not p: return ''
    parts=[]
    for f in ['overall_read','where_it_helps','where_it_could_mislead']:
        v=p.get(f,'')
        if isinstance(v,list): parts.extend(str(x) for x in v)
        elif isinstance(v,str): parts.append(v)
    return ' '.join(parts)

unframed={}
for f in os.listdir(S9):
    if not f.endswith('.json'): continue
    r=json.load(open(S9/f))
    if r.get('parse_succeeded'): unframed[(r['specimen_id'],r['model_id'])]=extract_unframed(r)
def load_framed(pid,fr,mdl):
    fp=S6/f"{pid}__{fr}__{mdl}.json"
    return extract_framed(json.load(open(fp))) if fp.exists() else None
models=sorted(set(k[1] for k in unframed))
print(f"Unframed clean: {len(unframed)} · models: {models}")

## S9b-1 — Embed + compute cosine to own F0/F1/F2

In [ ]:
# S9b-1
from sentence_transformers import SentenceTransformer
m=SentenceTransformer('all-MiniLM-L6-v2')
def emb(t): return m.encode([t],normalize_embeddings=True)[0]
def cos(a,b): return float(np.dot(a,b))

per_pair=[]
for (pid,mdl),uf in unframed.items():
    f0,f1,f2=load_framed(pid,'F0_neutral',mdl),load_framed(pid,'F1_clinical',mdl),load_framed(pid,'F2_metaphysical',mdl)
    if not all([f0,f1,f2]): continue
    ue=emb(uf); c0,c1,c2=cos(ue,emb(f0)),cos(ue,emb(f1)),cos(ue,emb(f2))
    per_pair.append({'principle':pid,'model':mdl,'cos_F0':round(c0,4),'cos_F1':round(c1,4),
                     'cos_F2':round(c2,4),'f1_minus_f2':round(c1-c2,4),
                     'closest':max([('F0',c0),('F1',c1),('F2',c2)],key=lambda x:x[1])[0]})
print(f"Analyzed {len(per_pair)} (principle,model) pairs")

## S9b-2 — Classify + threshold sensitivity

In [ ]:
# S9b-2
def classify(d,thr): return 'F1_clinical' if d>thr else ('F2_metaphysical' if d<-thr else 'balanced')
print("=== Per-model (threshold 0.03) ===")
for mdl in models:
    rs=[r for r in per_pair if r['model']==mdl]
    bal=sum(1 for r in rs if classify(r['f1_minus_f2'],0.03)=='balanced')
    l1=sum(1 for r in rs if classify(r['f1_minus_f2'],0.03)=='F1_clinical')
    l2=sum(1 for r in rs if classify(r['f1_minus_f2'],0.03)=='F2_metaphysical')
    f0c=sum(1 for r in rs if r['closest']=='F0')
    mn=np.mean([r['f1_minus_f2'] for r in rs])
    print(f"  {mdl:20} n={len(rs):2} balanced={bal:2} leanF1={l1:2} leanF2={l2:2} | closestF0={f0c:2} mean(F1-F2)={mn:+.4f}")

print("\n=== Threshold sensitivity (balanced fraction) ===")
for thr in [0.02,0.03,0.05]:
    bal=sum(1 for r in per_pair if classify(r['f1_minus_f2'],thr)=='balanced')
    print(f"  ±{thr}: {bal}/{len(per_pair)} balanced ({bal/len(per_pair):.3f})")
print(f"\nOverall mean(F1-F2): {np.mean([r['f1_minus_f2'] for r in per_pair]):+.4f}  (near 0 = no systematic population direction)")

## S9b-3 — Freeze artifact + verdict

In [ ]:
# S9b-3
def classify(d,thr): return 'F1_clinical' if d>thr else ('F2_metaphysical' if d<-thr else 'balanced')
sens={str(t):{'balanced':sum(1 for r in per_pair if classify(r['f1_minus_f2'],t)=='balanced'),
              'balanced_frac':round(sum(1 for r in per_pair if classify(r['f1_minus_f2'],t)=='balanced')/len(per_pair),3)} for t in [0.02,0.03,0.05]}
per_model={mdl:{'n':len([r for r in per_pair if r['model']==mdl]),
    'balanced':sum(1 for r in per_pair if r['model']==mdl and classify(r['f1_minus_f2'],0.03)=='balanced'),
    'lean_F1':sum(1 for r in per_pair if r['model']==mdl and classify(r['f1_minus_f2'],0.03)=='F1_clinical'),
    'lean_F2':sum(1 for r in per_pair if r['model']==mdl and classify(r['f1_minus_f2'],0.03)=='F2_metaphysical'),
    'closest_to_F0':sum(1 for r in per_pair if r['model']==mdl and r['closest']=='F0'),
    'mean_f1_minus_f2':round(float(np.mean([r['f1_minus_f2'] for r in per_pair if r['model']==mdl])),4)} for mdl in models}
overall={'n':len(per_pair),'balanced_at_0.03':sens['0.03']['balanced'],
    'balanced_frac_at_0.03':sens['0.03']['balanced_frac'],
    'mean_f1_minus_f2':round(float(np.mean([r['f1_minus_f2'] for r in per_pair])),4),
    'threshold_sensitivity':sens}
artifact={'analysis':'stage9b_unframed_baseline_test','study':'MTCA-2','hypothesis':'H10',
    'embedding_model':'all-MiniLM-L6-v2',
    'method':'per (principle,model): cosine(unframed_open_response, own F0/F1_clinical/F2_metaphysical); classify balanced/lean by |cos_F1-cos_F2| vs threshold',
    'n_pairs_analyzed':len(per_pair),
    'note_deepseek_p10':'SSP_P10/deepseek excluded (1 structural parse failure; 64/65 clean)',
    'overall':overall,'per_model':per_model,'per_pair':per_pair,
    'h10_verdict':'PARTIAL DISCONFIRMATION - balanced fraction threshold-dependent; near-zero population mean; per-model signatures robust; self-reports do not fully match measured behavior',
    'threshold_caveat':'0.03 cutoff is a modeling choice; continuous mean and sensitivity table reported',
    'language_discipline_note':'Model behavior under specified methodology. No claims about framework validity.'}
ser=json.dumps({k:v for k,v in artifact.items() if k!='per_pair'},sort_keys=True,ensure_ascii=False)
sha=hashlib.sha256(ser.encode()).hexdigest(); artifact['artifact_sha256']=sha
artifact['creation_date']=datetime.now(timezone.utc).isoformat()
outdir=MTCA2/"stage9b_analysis"; outdir.mkdir(parents=True,exist_ok=True)
(outdir/f"stage9b_unframed_test_{sha[:16]}.json").write_text(json.dumps(artifact,indent=2,ensure_ascii=False),encoding='utf-8')
print(f"Frozen: stage9b_unframed_test_{sha[:16]}.json\nSHA: {sha}")
print("\nVerdict: H10 partially DISCONFIRMED - models' Stage 8.5 self-prediction of 'synthesis' only partly matches measured unframed behavior.")